# Monte Carlo Pricing and Variance Reduction

This notebook prices vanilla, Asian and barrier options under GBM and compares Monte Carlo estimates with analytical Black-Scholes prices.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd

from derivatives_engine.models.black_scholes import call_price
from derivatives_engine.models.monte_carlo import compare_variance_reduction, price_asian_arithmetic_option, price_barrier_option, price_european_option, simulate_gbm_paths
from derivatives_engine.utils.plotting import plot_paths

In [ ]:
S, K, T, r, q, sigma = 100.0, 100.0, 1.0, 0.05, 0.0, 0.20
mc = price_european_option(S, K, T, r, q, sigma, 'call', n_paths=100_000, seed=123)
pd.DataFrame([{
    'mc_price': mc.price,
    'standard_error': mc.standard_error,
    'ci_lower': mc.confidence_interval[0],
    'ci_upper': mc.confidence_interval[1],
    'black_scholes': call_price(S, K, T, r, q, sigma),
}])

In [ ]:
compare_variance_reduction(S, K, T, r, q, sigma, 'call', n_paths=100_000, seed=789)

In [ ]:
asian = price_asian_arithmetic_option(S, K, T, r, q, sigma, 'call', n_paths=50_000, n_steps=126, seed=10)
barrier = price_barrier_option(S, K, T, r, q, sigma, 'call', barrier=130, barrier_type='up-and-out', n_paths=50_000, n_steps=126, seed=10)
pd.DataFrame([
    {'option': 'Asian arithmetic call', 'price': asian.price, 'standard_error': asian.standard_error},
    {'option': 'Up-and-out barrier call', 'price': barrier.price, 'standard_error': barrier.standard_error},
])

In [ ]:
paths = simulate_gbm_paths(S, T, r, q, sigma, n_paths=30, n_steps=126, seed=99)
time_grid = [i * T / 126 for i in range(127)]
plot_paths(time_grid, paths, 'GBM sample paths', max_paths=20)